validation data(400人)のみでWGCNAまで解析

# Preprocessing

# Import Library

In [ ]:
install.packages("tidyverse")

In [ ]:
library("tidyverse")
options(repr.plot.width=5, repr.plot.height=5)#Default=7x7
library(readxl)
library(dplyr)
library(ggplot2)
library(patchwork)

# Loading Datasets  


We import the following three primary datasets:

Omics Data: NPX values (expression intensity)

Sample Metadata: Clinical info (Age, Sex, BMI, etc.)

Peptide Count Data: Metrics for protein detection confidence.tics.


## Omic Data

In [ ]:
fileDir <- "./ssd/PE_R478/NPXfiles/converted/"
fileName <- "PE_R478_AN01_NPX_2026-04-01_converted.csv"

#All dataset
original_data <- read_delim(str_c(fileDir, fileName), delim = ",") %>%
    column_to_rownames(var = "OlinkID")


class(original_data)
print(str_c("nrow: ", nrow(original_data)))
head(original_data)
original_data[, 1:10] %>% summary()

## OIDとUniprotIDのmapping

In [ ]:
fileDir <- "./ssd/PE_R478/NPXfiles/original/"
fileName <- "/PE_R478_AN01_NPX_2026-04-01.csv"

temp <- read_delim(str_c(fileDir, fileName), delim = ";") 
head(temp)
temp2 <- temp %>% 
  select(OlinkID, UniProt) %>%
  distinct() %>%
  arrange(OlinkID)

head(temp2)
OID_Uni_map <- temp2

## Sample Metadata

NPX_PE_R478_179  
NPX_PE_R478_353  
を除外したメタデータを入力した

In [ ]:
fileDir <- "./ssd/PE_R478/NPXfiles/"
fileName <- "260701OlinkRevealPancreaticCancer398MetaData.csv"

#Sample info dataset
pro_sample <- read_delim(str_c(fileDir,fileName), delim = ",") 

class(pro_sample)
print(str_c("nrow: ", nrow(pro_sample)))
head(pro_sample)

In [ ]:
fileDir <- "./ssd/PE_R478/"
fileName <- "PE_R478_SampleID対応表.xlsx"

#Sample info dataset
temp <- read_excel(str_c(fileDir, fileName), skip = 3)

head(temp)
sample_map <- temp

# Data Preprocessing

## Sample名のNPXの部分を除外

In [ ]:
#only number dataset
tempDF <- original_data %>% 
    select(where(~ is.double(.)))

#transpose dataset
tempDF <- t(tempDF)
rownames(tempDF) <- gsub("NPX_", "", rownames(tempDF))
print(str_c("nrow: ", nrow(tempDF)))
head(tempDF)
pro_data <- tempDF

## pro_sampleの列名を英語にする

In [ ]:
temp <- pro_sample
colnames(temp)[colnames(temp) == "進展度"] <- "progress"
head(temp)
pro_sample <- temp

## NaN -> 0 (Progress row)

In [ ]:
temp <- pro_sample
temp[, "progress"][is.na(temp[, "progress"])] <- 0
head(temp)
pro_sample <- temp

## Progressを３段階に分ける

In [ ]:
temp <- pro_sample

temp <- temp %>%
  mutate(
    progress_group = case_when(
      progress == 0 ~ "0",
      progress %in% c(400, 410) ~ "1",
      progress %in% c(420, 430, 440) ~ "2",
      TRUE ~ NA_character_
    ),
    progress_group = factor(
      progress_group,
      levels = c("0", "1", "2")
    )
  )

table(temp$progress_group, useNA = "ifany")
class(temp$progress_group)
pro_sample <- temp

# Check missing values

## NAがないか確認

In [ ]:
head(pro_data)
any(!is.finite(as.matrix(pro_data)))

In [ ]:
head(pro_sample)
any(!is.finite(as.matrix(pro_sample)))
target_cols <- c("sex", "fvage", "py", "quant_drink_gr", "bmi", "hx_dm")

colSums(is.na(pro_sample[, target_cols, drop = FALSE]))

# 型チェック

In [ ]:
non_numeric_cols <- !sapply(temp, is.numeric)
sapply(temp[non_numeric_cols], class)

## Check sample data

In [ ]:
colSums(is.na(pro_sample))

がん患者と健常者で、進展度, icdo3t, icdo3mが正しく NA または値が入っているかを検証

In [ ]:
error_rows <- pro_sample %>%
  filter(
    (CaCo_Status == 1 & (is.na(progress_group) | is.na(icdo3t) | is.na(icdo3m))) |
    (CaCo_Status == 0 & (!is.na(progress_group) | !is.na(icdo3t) | !is.na(icdo3m)))
)

if (nrow(error_rows) == 0) {
  print("No Error")
} else {
  print(paste("Error row: ", nrow(error_rows)))
  error_rows %>% select(CaCo_Status, progress_group, icdo3t, icdo3m) %>% head()
}

## 除外されたサンプルが他のデータセットでも除外されているか確認

In [ ]:
check_missing_ids <- function(df, id_column) {
  
  # 指定された列からIDベクトルを取得し、末尾の数字を抽出
  ids <- df[[id_column]]
  present_numbers <- ids %>% 
    str_extract("\\d+$") %>%   
    as.numeric() %>% 
    unique() %>% 
    sort()
  
  # 1〜400の差分をチェック
  all_numbers <- 1:400
  missing_numbers <- setdiff(all_numbers, present_numbers)
  
  # 結果をリスト形式で返却
  return(list(
    is_complete = (length(missing_numbers) == 0),
    missing_count = length(missing_numbers),
    missing_numbers = missing_numbers
  ))
}

temp <- pro_data %>% 
  as.data.frame() %>%
  rownames_to_column(var = "SampleID")
result1 <- check_missing_ids(temp, "SampleID")
print(result1$missing_numbers)

result2 <- check_missing_ids(pro_sample, "Sample ID")
print(result2$missing_numbers)

result3 <- check_missing_ids(sample_map, "RG Sample ID")
print(result3$missing_numbers)

## 除外されていないものを除外

In [ ]:
temp <- c("PE_R478_179", "PE_R478_353")

temp2 <- sample_map %>% 
  filter(!(`RG Sample ID` %in% temp))

result1 <- check_missing_ids(temp2, "RG Sample ID")
print(result1$missing_numbers)

sample_map <- temp2

## Check omics data

### 分散が0のものをみる

In [ ]:
temp <- pro_data
zero_var_cols <- apply(temp, 2, function(x) is.numeric(x) && var(x, na.rm = TRUE) == 0)
colnames(temp)[zero_var_cols & !is.na(zero_var_cols)]

In [ ]:
summary(temp[, "OID50003"])
var(temp[, "OID50003"], na.rm = TRUE)

In [ ]:
temp <- pro_data
temp <- temp[, colnames(temp) != "OID50003", drop = FALSE]
head(temp)

pro_data <- temp

## Export Sample information data file

In [ ]:
if (!dir.exists("./Data")) {
  dir.create("./Data")
}

write.csv(pro_data, file = "Data/pro_data_val.csv", row.names = TRUE)
write.csv(OID_Uni_map, file = "Data/map_OID_Uni.csv", row.names = TRUE)
write.csv(pro_sample, file = "Data/pro_sample_val.csv", row.names = FALSE)

In [ ]:
if (!dir.exists("./ExportData")) {
  dir.create("./ExportData")
}


# データの概要を見る

In [ ]:
temp <- pro_sample
head(temp)
cat_cols <- c("sex", "fvage", "py", "quant_drink_gr", "bmi", "hx_dm", "CaCo_Status", "progress_group")

temp_summary <- temp[, cat_cols, drop = FALSE]

factor_cols <- c("sex", "hx_dm", "CaCo_Status", "progress_group")
temp_summary[factor_cols] <- lapply(temp_summary[factor_cols], as.factor)

int_cols <- c("fvage", "py", "quant_drink_gr", "bmi")
temp_summary[int_cols] <- lapply(temp_summary[int_cols], as.integer)

summary(temp_summary)

# UniprotIDとProtein Symbolsの表を作成

In [ ]:
head(OID_Uni_map)

Entry nameとAccessionが混在している？

## Entry nameとaccessionを分類

In [ ]:
uniprot_list <- as.character(OID_Uni_map$UniProt)

# Accessino pattern
ac_regex <- "^([OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2})$"
is_accession <- grepl(ac_regex, uniprot_list, ignore.case = TRUE)

accessions <- uniprot_list[is_accession]
others     <- uniprot_list[!is_accession]

cat("Accession name: ", head(accessions), ", num: ", length(accessions),"\n")
cat("Others: ", others, ", num: ", length(others),"\n")

In [ ]:
entry <- others[others %in% c("AMP1", "INC1", "EXT1")]
head(entry)

In [ ]:
writeLines(accessions, "./ExportData/uniprot_accessions.txt")
writeLines(as.character(OID_Uni_map$UniProt), "./ExportData/uniprot_ids.txt")

accessionをhttps://www.uniprot.org/id-mapping でGene nameに変換  
出力結果をDataに保存

In [ ]:
#PancreasProject/Aichi_Proteomics/Data/idmapping_2026_07_17.tsv.gz
temp <- read_tsv("./Data/idmapping_2026_07_17.tsv.gz")
head(temp)
# Left join and handle missing values
temp2 <- OID_Uni_map %>%
  left_join(temp, by = c("UniProt" = "From")) %>%
  mutate(Symbol = if_else(is.na(To), paste0(UniProt, "*"), To)) %>%
  select(-To)

# Preview result
head(temp2)
val_map <- temp2
write.csv(val_map, file = "Data/map_OID_Uni_Sym.csv", row.names = TRUE)

# Discoveryとのオミクスの重複を見る

In [ ]:
disco <- read_csv("../Proteomics/Data/pro_analyte.csv")
colnames(disco)[1] <- "OID"
head(disco)

disco_uniprot <- unique(na.omit(disco$UniProtID))
val_uniprot   <- unique(na.omit(val_map$UniProt))
overlapping_ids <- intersect(disco_uniprot, val_uniprot)

# Calculate overlap counts
overlap_count  <- length(overlapping_ids)
val_only_count <- length(setdiff(val_uniprot, disco_uniprot))

# Extract IDs only in val_map
val_only_ids <- setdiff(val_uniprot, disco_uniprot)
# Filter val_map using the unmatched IDs vector
val_only_symbols <- val_map %>%
  filter(UniProt %in% val_only_ids) %>%
  select(UniProt, Symbol) %>%
  distinct()


cat("Number of overlapping IDs:", overlap_count, "\n")
cat("Number of IDs in val_map but not in disco:", val_only_count, " (",val_only_ids,")\n")


# Preview the specific IDs missing in disco
head(val_only_symbols)

## 重複しているもののみを保存する

## Validation dataの方

In [ ]:
# Map overlapping UniProt IDs to OIDs
target_oids <- val_map %>%
  filter(UniProt %in% overlapping_ids) %>%
  pull(OlinkID) %>%
  unique() %>%
  na.omit()

pro_data_filtered <- pro_data[, colnames(pro_data) %in% target_oids]

oid_to_uniprot <- setNames(val_map$UniProt, val_map$OlinkID)
colnames(pro_data_filtered) <- oid_to_uniprot[colnames(pro_data_filtered)]

cat("Number of target OIDs found:", length(target_oids), "\n")
cat("Number of rows extracted from pro_data:", ncol(pro_data_filtered), "\n")
head(pro_data_filtered)

In [ ]:
write.csv(pro_data_filtered, file = "./Data/pro_data_common_val.csv", row.names = TRUE)

## Discovery dataの方

In [ ]:
head(disco)

In [ ]:
pro_data_dis <- read.csv("../Proteomics/Data/pro_data.csv",row.names = 1)
# colnames(pro_data_dis)[1] <- "OID"
head(pro_data_dis)

# Map overlapping UniProt IDs to OIDs
target_oids <- disco %>%
  filter(UniProtID %in% overlapping_ids) %>%
  pull(OID) %>%
  unique() %>%
  na.omit()

pro_data_filtered_dis <- pro_data_dis[, colnames(pro_data_dis) %in% target_oids]

oid_to_uniprot <- setNames(disco$UniProtID, disco$OID)
colnames(pro_data_filtered_dis) <- oid_to_uniprot[colnames(pro_data_filtered_dis)]

cat("Number of target OIDs found:", length(target_oids), "\n")
cat("Number of rows extracted from pro_data:", ncol(pro_data_filtered_dis), "\n")
head(pro_data_filtered_dis)

In [ ]:
pro_sample_dis <- read.csv("../Proteomics/Data/pro_sample.csv",row.names = 1)
head(pro_sample_dis)
dim(pro_sample_dis)

In [ ]:
write.csv(pro_sample_dis, file = "./Data/pro_sample_dis.csv", row.names = TRUE)

In [ ]:
write.csv(pro_data_filtered_dis, file = "./Data/pro_data_common_dis.csv", row.names = TRUE)

In [ ]:
write.csv(overlapping_ids, file = "./Data/overlapping_ids.csv", row.names = TRUE)

In [ ]:
map <- read.csv("./Data/map_OID_Uni_Sym.csv", row.names = 1)

# UniProt → Symbol
uni_to_symbol <- setNames(map$Symbol, map$UniProt)

# pro_data_disのUniProt列名を取得
uniprot_names <- colnames(pro_data_filtered_dis)

# Symbolに変換
symbol_names <- ifelse(
  uniprot_names %in% names(uni_to_symbol),
  uni_to_symbol[uniprot_names],
  uniprot_names
)

# 重複しているSymbolを確認
dup_symbols <- unique(symbol_names[duplicated(symbol_names)])

# 重複しているSymbolだけ、Symbol(UniProt)にする
for (i in seq_along(symbol_names)) {
  if (symbol_names[i] %in% dup_symbols) {
    symbol_names[i] <- paste0(
      symbol_names[i],
      "(",
      uniprot_names[i],
      ")"
    )
  }
}

temp <- pro_data_filtered_dis
# 列名を変更
colnames(temp) <- symbol_names
head(temp)
colnames(temp)[grepl("^PALM2AKAP2", colnames(pro_data_dis))]
pro_data_filtered_dis <- temp
write.csv(pro_data_filtered_dis, file = "./Data/pro_data_common_dis_Assay.csv", row.names = TRUE)